In [ ]:
#| hide
#| eval: false
! [ -e /content ] && pip install -Uqq fastai  # upgrade fastai on colab

In [ ]:
#| export
from fastai.basics import *

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| default_exp callback.rnn

# Callback for RNN training

> Callback that uses the outputs of language models to add AR and TAR regularization

In [ ]:
#| export
@docs
class ModelResetter(Callback):
    "`Callback` that resets the model at each validation/training step"
    def before_train(self):    self.model.reset()
    def before_validate(self): self.model.reset()
    def after_fit(self):       self.model.reset()
    _docs = dict(before_train="Reset the model before training",
                 before_validate="Reset the model before validation",
                 after_fit="Reset the model after fitting")

In [ ]:
#| export
class RNNCallback(Callback):
    "Save the raw and dropped-out outputs and only keep the true output for loss computation"
    def after_pred(self): self.learn.pred,self.raw_out,self.out = [o[-1] if is_listy(o) else o for o in self.pred]

In [ ]:
#| export
class RNNRegularizer(Callback):
    "Add AR and TAR regularization"
    order,run_valid = RNNCallback.order+1,False
    def __init__(self, alpha=0., beta=0.): store_attr()
    def after_loss(self):
        if not self.training: return
        if self.alpha: self.learn.loss_grad += self.alpha * self.rnn.out.float().pow(2).mean()
        if self.beta:
            h = self.rnn.raw_out
            if h.shape[1]>1: self.learn.loss_grad += self.beta * (h[:,1:] - h[:,:-1]).float().pow(2).mean()

In [ ]:
class _RegularizerModel(Module):
    def __init__(self, as_list=False):
        self.weight,self.as_list = nn.Parameter(tensor(1.)),as_list
    def forward(self, x):
        h = self.weight*x
        out = 2*h
        return h,([h] if self.as_list else h),([out] if self.as_list else out)

class _CheckRegularizer(Callback):
    order = RNNRegularizer.order+1
    def __init__(self, alpha, beta): store_attr()
    def after_loss(self):
        expected = self.pred.square().mean()
        if self.training:
            expected = expected + self.alpha*(2*self.pred).square().mean()
            if self.pred.shape[1]>1:
                expected = expected + self.beta*(self.pred[:,1:]-self.pred[:,:-1]).square().mean()
        test_close(self.loss_grad, expected)
        if self.training: self.expected_grad = 2*expected.detach()/self.model.weight.detach()
    def after_backward(self): test_close(self.model.weight.grad, self.expected_grad)
    def before_step(self): self.expected_weight = self.model.weight.detach().clone() - self.opt.hypers[0]['lr']*self.expected_grad
    def after_step(self): test_close(self.model.weight, self.expected_weight)

# Include singleton batches, singleton sequences, and a final partial batch.
for n,bs,sl in [(1,1,3), (2,2,3), (2,2,1), (1,1,1), (3,2,3)]:
    for alpha,beta in [(0.,1.), (0.,0.), (.5,1.)]:
        for as_list in [False,True]:
            x = torch.arange(1,sl+1).float().view(1,sl,1).repeat(n,1,1)
            dl = DataLoader(list(zip(x,torch.zeros_like(x))), bs=bs, num_workers=0)
            dls = DataLoaders(dl,dl,device=torch.device('cpu'))
            learn = Learner(dls, _RegularizerModel(as_list), loss_func=MSELossFlat(), opt_func=SGD,
                            cbs=[RNNCallback(), RNNRegularizer(alpha,beta), _CheckRegularizer(alpha,beta)], wd=0)
            with learn.no_logging(): learn.fit(1, lr=.01)


In [ ]:
#| export
def rnn_cbs(alpha=0., beta=0.):
    "All callbacks needed for (optionally regularized) RNN training"
    reg = [RNNRegularizer(alpha=alpha, beta=beta)] if alpha or beta else []
    return [ModelResetter(), RNNCallback()] + reg

## Export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()